In [ ]:
!apt-get install -y libmagic-dev poppler-utils tesseract-ocr
!pip install -q langchain-groq langchain-community langchain-huggingface langchain-text-splitters faiss-cpu langgraph typing_extensions openpyxl pandas gdown

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
libmagic-dev is already the newest version (1:5.41-3ubuntu0.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.


In [ ]:
import os
import pandas as pd
import gdown
from langchain_core.documents import Document

os.environ["GROQ_API_KEY"] = "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"

# --- 2. СКАЧИВАНИЕ МЕНЮ ---
print(">>> Загрузка меню...")
file_id = '1pH77UgDt1t2NFsrLb04YeC8cDP1nnN17'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'dim sum montijo.xlsx'

if not os.path.exists(output):
    gdown.download(url, output, quiet=True)

# --- 3. ЧТЕНИЕ ФАЙЛА ---
df = pd.read_excel(output)
documents = []

for index, row in df.iterrows():
    parts = []
    # Собираем данные в одну строку для базы знаний
    if pd.notna(row.get('ItemNameEn')): parts.append(f"Dish: {row['ItemNameEn']}")
    if pd.notna(row.get('ItemDescriptionEn')): parts.append(f"Ingredients: {row['ItemDescriptionEn']}")
    if pd.notna(row.get('ItemPrice')): parts.append(f"Price: {row['ItemPrice']} EUR")

    if parts:
        full_text = ". ".join(parts)
        documents.append(Document(page_content=full_text))

print(f"✅ Меню обработано: {len(documents)} позиций.")

>>> Загрузка меню...
✅ Меню обработано: 52 позиций.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

# 1. Поиск (Embeddings) - Бесплатный, локальный, мультиязычный
print(">>> Инициализация поиска...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
db = FAISS.from_documents(documents, embeddings)

# 2. LLM (Groq) - Используем мощную модель из вашего списка
print(">>> Подключение к Groq (Llama 3.3)...")
llm = ChatGroq(
    temperature=0.2,
    model_name="llama-3.3-70b-versatile"
)
print("✅ Модели готовы.")

>>> Инициализация поиска...
>>> Подключение к Groq (Llama 3.3)...
✅ Модели готовы.


In [ ]:
from typing import List
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate

# --- СОСТОЯНИЕ ---
class AgentState(TypedDict):
    question: str
    answer: str
    topic: str
    documents: str

# --- СЦЕНАРИЙ ДИАЛОГА ---

simulated_dialogue = [
    "Hi! Do you have any soups on the menu?",
    "How much is the Won Ton Soup?",
    "I'd like to try some dumplings. Tell me about the Siao Long Pao.",
    "Do you have any vegetarian options for Gyozas?",
    "I am thirsty. What kind of beer do you have?",
    "What is the weather like in London?", # Вопрос не по теме (Off-topic)
    "Back to food. What Mochi flavors do you have?",
    "I will have the Peanut Mochi. Check please!"
]

dialogue_iter = iter(simulated_dialogue)

def get_next_input():
    try:
        return next(dialogue_iter)
    except StopIteration:
        return "exit"

# --- ФУНКЦИИ (УЗЛЫ ГРАФА) ---

def greetings(state):
    print("\n" + "="*60)
    print("🤖 АГЕНТ: Добро пожаловать! Я ваш официант.")
    user_input = get_next_input()
    print(f"👤 КЛИЕНТ: {user_input}")
    return {"question": user_input}

def check_question(state):
    q = state['question'].lower()

    # 1. Проверка на счет
    if any(w in q for w in ["счет", "счёт", "чек", "оплатить"]):
        return {"topic": "bill"}

    # 2. Проверка по ключевым словам (чтобы наверняка понять блюда)
    keywords = ["суп", "вон тон", "сяо лонг", "гёдза", "моти", "пиво", "меню", "еда", "цена", "стоит"]
    if any(w in q for w in keywords):
        return {"topic": "on_topic"}

    # 3. Умная проверка через LLM
    try:
        system = "You are a classifier. Is the user question related to restaurant/food/drinks/menu? Answer ONLY 'yes' or 'no'."
        prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{question}")])
        chain = prompt | llm
        response = chain.invoke({"question": state['question']}).content.strip().lower()

        if "no" in response:
            return {"topic": "off_topic"}
    except:
        pass # Если сбой сети, считаем что вопрос по теме

    return {"topic": "on_topic"}

def retrieve_docs(state):
    docs = db.similarity_search(state['question'], k=3)
    context = "\n".join([f"- {d.page_content}" for d in docs])
    return {"documents": context}

def generate(state):
    system = """You are a polite waiter.
    Use the provided Menu Context to answer the user's question.
    ALWAYS answer in RUSSIAN.
    If you don't know the answer based on the menu, politely say so."""

    human = "Menu Context:\n{context}\n\nUser Question: {question}"

    prompt = ChatPromptTemplate.from_messages([("system", system), ("human", human)])
    chain = prompt | llm

    response = chain.invoke({"context": state['documents'], "question": state['question']})
    print(f"🤖 АГЕНТ: {response.content}")
    return {"answer": response.content}

def off_topic_response(state):
    msg = "Извините, я могу говорить только о нашем меню."
    print(f"🤖 АГЕНТ: {msg}")
    return {"answer": msg}

def process_bill(state):
    msg = "Конечно! Несу ваш счет. Спасибо за визит!"
    print(f"🤖 АГЕНТ: {msg}")
    return {"answer": msg}

def further_question(state):
    if state.get('topic') == "bill": return {"question": "exit"}
    user_input = get_next_input()
    if user_input == "exit": return {"question": "exit"}
    print(f"\n👤 КЛИЕНТ: {user_input}")
    return {"question": user_input}

In [ ]:
from langgraph.graph import StateGraph, END

def topic_router(state): return state['topic']

def should_continue(state):
    if state['question'] == "exit":
        return END
    return "check_question"

# Создаем граф
workflow = StateGraph(AgentState)

# Добавляем узлы
workflow.add_node("greetings", greetings)
workflow.add_node("check_question", check_question)
workflow.add_node("retrieve_docs", retrieve_docs)
workflow.add_node("generate", generate)
workflow.add_node("off_topic", off_topic_response)
workflow.add_node("bill", process_bill)
workflow.add_node("next", further_question)

# Точка входа
workflow.set_entry_point("greetings")

# --- СВЯЗИ (EDGES) ---
workflow.add_edge("greetings", "check_question") # Исправленная связь
workflow.add_conditional_edges("check_question", topic_router, {
    "on_topic": "retrieve_docs",
    "off_topic": "off_topic",
    "bill": "bill"
})
workflow.add_edge("retrieve_docs", "generate")
workflow.add_edge("generate", "next")
workflow.add_edge("off_topic", "next")
workflow.add_edge("bill", "next")
workflow.add_conditional_edges("next", should_continue)

app = workflow.compile()
print("✅ Граф успешно собран.")

✅ Граф успешно собран.


In [ ]:
import asyncio

print(">>> ЗАПУСК ДИАЛОГА...")

# Увеличиваем лимит рекурсии до 100 шагов (чтобы хватило на весь диалог)
config = {"recursion_limit": 100}

try:
    # Используем await, так как в Colab это работает стабильнее
    await app.ainvoke({"question": ""}, config=config)
except Exception as e:
    print(f"\nДиалог завершен (или ошибка: {e})")

>>> ЗАПУСК ДИАЛОГА...

🤖 АГЕНТ: Добро пожаловать! Я ваш официант.
👤 КЛИЕНТ: Здравствуйте! У вас есть супы?
🤖 АГЕНТ: Да, у нас есть супы. У нас есть два вида супов: Суп Вон Тон, приготовленный с паксоем, свининой и креветками, и Овощной суп, приготовленный с паксоем, бамбуком, грибами и овощами. Оба супа стоят 3,95 евро. Хотите ли вы заказать один из них?

👤 КЛИЕНТ: Сколько стоит суп Вон Тон?
🤖 АГЕНТ: Суп Вон Тон стоит 3,95 евро.

👤 КЛИЕНТ: Расскажите про Сяо Лонг Бао.
🤖 АГЕНТ: Традиционный Сяо Лонг Бао - это одно из наших популярных блюд. Он состоит из свинины, паксоя и шиитаке грибов. Цена этого блюда составляет 4,50 евро. Хотите ли вы заказать его?

👤 КЛИЕНТ: Есть ли вегетарианские Гёдза?
🤖 АГЕНТ: К сожалению, в нашем меню нет вегетарианских вариантов Гёдза. Все представленные блюда содержат свинину. Если вы ищете вегетарианский вариант, я могу предложить вам другие блюда из нашего меню, но, к сожалению, Гёдза у нас только с мясом. Хотите, я порекомендую вам что-то другое?

👤 КЛИЕНТ:

In [ ]:
import asyncio

print(">>> ЗАПУСК ДИАЛОГА...")

# Увеличиваем лимит рекурсии до 100 шагов (чтобы хватило на весь диалог)
config = {"recursion_limit": 100}

try:
    # Используем await, так как в Colab это работает стабильнее
    await app.ainvoke({"question": ""}, config=config)
except Exception as e:
    print(f"\nДиалог завершен (или ошибка: {e})")

>>> ЗАПУСК ДИАЛОГА...

🤖 АГЕНТ: Добро пожаловать! Я ваш официант.
👤 КЛИЕНТ: Hi! Do you have any soups on the menu?
🤖 АГЕНТ: Да, у нас есть два вида супов в меню. У нас есть Суп Вон Тон, приготовленный с паксоем, свининой и креветками, и вегетарианский суп, приготовленный с паксоем, бамбуком, грибами и овощами. Оба супа стоят 3,95 евро. Хотите ли вы заказать один из них?

👤 КЛИЕНТ: How much is the Won Ton Soup?
🤖 АГЕНТ: Суп Вон Тон стоит 3,95 евро.

👤 КЛИЕНТ: I'd like to try some dumplings. Tell me about the Siao Long Pao.
🤖 АГЕНТ: Добрый день! У нас есть два вида традиционных Сяо Лун Бао. Первый - Традиционный Сяо Лун Бао, приготовленный с свининой, паксоем и шиитаке грибами. Второй - Чёрный Сяо Лун Бао, в состав которого входят свинина, имбирь и лук. Оба варианта стоят 4,50 евро. Какой из них вам больше понравится?

👤 КЛИЕНТ: Do you have any vegetarian options for Gyozas?
🤖 АГЕНТ: Да, у нас есть вегетарианский вариант гёдза - Vegetable Gyoza, приготовленный из овощей. Его цена составл